---
# How This Code Works
---
This notebook is a guided tour of the current tutorial codebase.

It is written for readers who want to understand the structure of the project without reverse-engineering how the files fit together. The emphasis is on the *roles* of the files, classes, and helper functions, and on how the notebooks call into the shared Python modules.


---
## Table of Contents

1. [Big Picture](#big-picture)
2. [Imports and Project Root](#imports)
3. [Codebase Map](#file-map)
4. [Configuration Objects](#config)
5. [Inventory MDP Layer](#mdp)
6. [Basis Functions](#basis)
7. [Model Classes](#models)
8. [Evaluation Helpers](#evaluation)
9. [Orchestration Helpers](#helpers)
10. [Notebook Workflow](#workflow)
11. [Dependency Picture](#dependencies)
12. [Reading Order](#reading-order)


---
<a id="big-picture"></a>
## 1. Big Picture

This project studies one inventory-control problem with three approximate dynamic programming approaches:

- `FALP`: fit one approximate linear program for a chosen basis size.
- `SGALP`: grow the basis gradually and add guiding constraints between stages.
- `PSMD`: use a lighter stochastic projected-gradient baseline.

Across all three methods, the code follows the same pattern:

1. define the inventory model in `mdp.py`
2. define an approximation family in `basis.py`
3. fit coefficients with one of the model classes
4. estimate a lower bound with `self_guided_alp/cvl_lower_bound.py`
5. estimate policy cost with `policy.py`
6. organize experiments and figures with `helper.py`
7. explain the outputs in the notebooks inside `notebooks/`


In [ ]:
import sys
from dataclasses import fields
from pathlib import Path
import inspect


def find_project_root(start_path: Path) -> Path:
    """
    Find the tutorial project root by looking for the shared Python modules.

    Args:
        start_path: Directory from which to begin the upward search.
    """
    for candidate in (start_path, *start_path.parents):
        if (candidate / "helper.py").exists() and (candidate / "config.py").exists():
            return candidate
    raise RuntimeError("Could not locate the tutorial project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from basis import PolynomialBasis1D, RandomFourierBasis1D
from config import (
    FALPConfig,
    GuidingConstraintConfig,
    HiGHSSolverConfig,
    InventoryMDPConfig,
    LowerBoundConfig,
    PolicyEvaluationConfig,
    PSMDConfig,
    RandomFeatureConfig,
    SGALPConfig,
)
from helper import (
    apply_tutorial_plot_style,
    make_shared_evaluation_configs,
    plot_bound_boxplots,
    plot_falp_vs_sgalp_policy_costs,
    plot_falp_vs_sgalp_vfas_and_relevance,
    plot_psmd_acceptance_and_value,
    plot_psmd_iteration_diagnostics,
    plot_psmd_sampling_snapshots,
    plot_value_function_curves,
    run_falp_and_sgalp_comparison,
    run_falp_grid,
    run_psmd_seed_grid,
    run_sgalp_grid,
    run_sgalp_stage_trace,
    summarize_falp_vs_sgalp_policy_costs,
)
from mdp import MarkovDecisionProcess, SingleProductInventoryMDP, UniformSampler, make_inventory_mdp
from policy import build_greedy_policy_lookup, estimate_upper_bound_fast
from psmd.psmd import PSMD
from self_guided_alp.cvl_lower_bound import (
    SimpleLNSLowerBound,
    estimate_actual_lower_bound_falp,
    estimate_actual_lower_bound_sgalp,
)
from self_guided_alp.falp import FALP
from self_guided_alp.sgalp import SelfGuidedALP

apply_tutorial_plot_style()
print('Project root:', PROJECT_ROOT)


---
<a id="file-map"></a>
## 3. Codebase Map

The project is intentionally small. Most of the logic lives in a handful of modules.


In [ ]:
print('Root Python modules:')
for path in sorted(PROJECT_ROOT.glob('*.py')):
    print('  ', path.name)

print('\nself_guided_alp package:')
for path in sorted((PROJECT_ROOT / 'self_guided_alp').glob('*.py')):
    print('  ', f'self_guided_alp/{path.name}')

print('\nPSMD package:')
for path in sorted((PROJECT_ROOT / 'psmd').glob('*.py')):
    print('  ', f'psmd/{path.name}')

print('\nNotebooks:')
for path in sorted((PROJECT_ROOT / 'notebooks').glob('*.ipynb')):
    print('  ', f'notebooks/{path.name}')


The roles of the main files are:

| Path | Main role | Why it matters |
| --- | --- | --- |
| `config.py` | grouped parameter objects | keeps constructors and helper calls readable |
| `mdp.py` | discounted inventory model | defines state dynamics, costs, and sampling routines |
| `basis.py` | value-function basis families | provides the approximation architecture |
| `self_guided_alp/falp.py` | FALP solver | fits one approximate LP |
| `self_guided_alp/sgalp.py` | SGALP solver | fits a staged sequence with guiding constraints |
| `self_guided_alp/cvl_lower_bound.py` | lower-bound estimation | gives a performance certificate |
| `policy.py` | greedy-policy simulation | estimates policy cost under a fitted approximation |
| `psmd/psmd.py` | PSMD baseline | provides a compact stochastic-gradient comparator |
| `helper.py` | tutorial orchestration | runs seed grids, builds plots, and packages experiments |
| `notebooks/*.ipynb` | explanations and demos | show how the building blocks are used together |


---
<a id="config"></a>
## 4. Configuration Objects

One of the biggest refactors in this codebase is the move toward grouped config objects.

Instead of passing many unrelated scalars through constructors, the project now uses dataclasses to bundle related choices together. This makes the notebooks easier to read and makes it clearer which parameters belong to the model, the solver, the lower-bound estimator, or the policy simulation.


In [ ]:
config_classes = [
    RandomFeatureConfig,
    HiGHSSolverConfig,
    GuidingConstraintConfig,
    FALPConfig,
    SGALPConfig,
    LowerBoundConfig,
    PolicyEvaluationConfig,
    PSMDConfig,
    InventoryMDPConfig,
]

for config_class in config_classes:
    print(f'\n{config_class.__name__}')
    print('-' * len(config_class.__name__))
    for field in fields(config_class):
        print(f'  {field.name}')


A useful way to read these config classes is by purpose:

- `RandomFeatureConfig` controls how the random Fourier basis is sampled.
- `HiGHSSolverConfig` stores numerical settings for the linear-program solver.
- `GuidingConstraintConfig` stores SGALP-only settings.
- `FALPConfig`, `SGALPConfig`, and `PSMDConfig` store algorithm-level settings.
- `LowerBoundConfig` and `PolicyEvaluationConfig` store *evaluation* settings shared across notebooks.
- `InventoryMDPConfig` stores the actual inventory-model data.

The helper function `make_shared_evaluation_configs(...)` is especially important in the notebooks because it keeps the lower-bound and policy-cost settings aligned across FALP, SGALP, and PSMD.


In [ ]:
shared_lower_bound_config, shared_policy_config = make_shared_evaluation_configs(initial_state=0.0)
print(shared_lower_bound_config)
print()
print(shared_policy_config)


---
<a id="mdp"></a>
## 5. Inventory MDP Layer

`mdp.py` has two jobs:

1. define a very small abstract `MarkovDecisionProcess` interface
2. implement the concrete inventory model used everywhere else

That second part is the important one for the tutorial. `SingleProductInventoryMDP` defines:

- the inventory state
- the order quantity action
- the stochastic demand process
- the one-period cost
- the sampling routines reused by ALP fitting, lower-bound estimation, and policy simulation


In [ ]:
mdp = make_inventory_mdp()
print('MDP class:', type(mdp).__name__)
print('State bounds:', (mdp.lower_state_bound, mdp.upper_state_bound))
print('Max order:', mdp.max_order)
print('Discount factor:', mdp.discount)
print('Discrete actions:', mdp.get_discrete_actions()[:6], '...')
print('Noise sample:', mdp.sample_noise_batch(num_samples=5, random_seed=2026))


Two design choices make the rest of the code simpler:

- The MDP exposes vectorized routines such as `evaluate_state_action_batch(...)`, so the policy and PSMD code can reuse one shared transition-and-cost interface.
- The MDP also keeps a few backward-compatible aliases so the notebooks still read naturally while the internal names become more descriptive.


---
<a id="basis"></a>
## 6. Basis Functions

`basis.py` defines the approximation families used by the models.

There are two basis classes:

- `RandomFourierBasis1D`: used by FALP and SGALP
- `PolynomialBasis1D`: used by the lightweight PSMD baseline

The key conceptual point is that FALP and SGALP use the *same* random-feature family. Their difference is not the basis. Their difference is how they impose and update constraints.


In [ ]:
rf_basis = RandomFourierBasis1D(
    max_random_features=4,
    config=RandomFeatureConfig(bandwidth_choices=(1e-2, 1e-5), random_seed=111),
)
poly_basis = PolynomialBasis1D(exponents=(0, 1, 2))

state_grid = [-2.0, 0.0, 3.0]
print('Random Fourier basis matrix shape:', rf_basis.eval_basis_batch(state_grid, num_random_features=2).shape)
print(rf_basis.eval_basis_batch(state_grid, num_random_features=2))
print()
print('Polynomial basis matrix shape:', poly_basis.eval_basis_batch(state_grid).shape)
print(poly_basis.eval_basis_batch(state_grid))


For non-experts, one helpful interpretation is:

- the basis turns a state `s` into a feature vector
- the model learns coefficients for those features
- the approximate value function is just “basis values times coefficients”

That is why almost every fitted model in the project exposes the same trio of ingredients: `mdp`, `basis`, and `coef`.


---
<a id="models"></a>
## 7. Model Classes

The project has three main model classes.

| Class | File | Main idea |
| --- | --- | --- |
| `FALP` | `self_guided_alp/falp.py` | fit one approximate LP at one basis size |
| `SelfGuidedALP` | `self_guided_alp/sgalp.py` | grow the basis in stages and add guiding constraints |
| `PSMD` | `psmd/psmd.py` | run projected stochastic updates on a compact surrogate |

All three classes operate on the same inventory MDP, but they fit the approximation in different ways.


In [ ]:
for cls in [FALP, SelfGuidedALP, PSMD, SimpleLNSLowerBound]:
    print(f'\n{cls.__name__}')
    print('  __init__ signature:', inspect.signature(cls.__init__))


A simple way to separate their responsibilities is:

- `FALP` and `SelfGuidedALP` are *fitters* for the ALP-style approximations.
- `SimpleLNSLowerBound` is an *evaluator* that takes a fitted approximation and estimates a lower bound.
- `PSMD` is both a fitter and its own tutorial baseline, but it still relies on the shared lower-bound and policy-evaluation components.


---
<a id="evaluation"></a>
## 8. Evaluation Helpers

After fitting a model, the project reports two complementary diagnostics:

- a **lower bound** from `self_guided_alp/cvl_lower_bound.py`
- a **policy cost** from `policy.py`

The lower bound is a performance certificate. The policy cost is a simulation-based estimate of what the induced greedy policy actually costs.


In [ ]:
print('build_greedy_policy_lookup:', inspect.signature(build_greedy_policy_lookup))
print('estimate_upper_bound_fast:', inspect.signature(estimate_upper_bound_fast))
print('estimate_actual_lower_bound_falp:', inspect.signature(estimate_actual_lower_bound_falp))
print('estimate_actual_lower_bound_sgalp:', inspect.signature(estimate_actual_lower_bound_sgalp))


The evaluation story is intentionally shared across the notebooks:

- FALP and SGALP both call the same policy-evaluation logic.
- PSMD also uses the same `estimate_upper_bound_fast(...)` helper on a small fitted-model view.
- The notebooks usually create `shared_lower_bound_config` and `shared_policy_config` once and reuse them throughout so comparisons stay apples-to-apples.


---
<a id="helpers"></a>
## 9. Orchestration Helpers

`helper.py` is the notebook support layer. It does not define the inventory problem or the approximation theory. Instead, it packages repetitive experiment logic into reusable helpers with shorter, more readable calls.


In [ ]:
helper_groups = {
    'small utilities': [
        'apply_tutorial_plot_style',
        'make_shared_evaluation_configs',
        'evaluate_vfa_on_grid',
        'estimate_cvl_lower_bound',
    ],
    'experiment runners': [
        'run_falp_grid',
        'run_sgalp_grid',
        'run_psmd_seed_grid',
        'run_sgalp_stage_trace',
        'run_falp_and_sgalp_comparison',
    ],
    'plot helpers': [
        'plot_value_function_curves',
        'plot_bound_boxplots',
        'plot_psmd_iteration_diagnostics',
        'plot_psmd_acceptance_and_value',
        'plot_psmd_sampling_snapshots',
        'plot_falp_vs_sgalp_vfas_and_relevance',
        'plot_falp_vs_sgalp_policy_costs',
    ],
}

for group_name, names in helper_groups.items():
    print(f'\n{group_name}')
    print('-' * len(group_name))
    for name in names:
        print(' ', name)


A few recent changes are worth noticing here:

- the helper layer now uses grouped config objects much more consistently
- FALP, SGALP, and PSMD plotting goes through shared plotting style defaults
- the FALP-versus-SGALP comparison logic is centralized instead of being duplicated inside the notebook
- boxplots now use a shared style with visible `+` outliers and a black min-to-max line


---
<a id="workflow"></a>
## 10. Notebook Workflow

The public notebooks are not meant to reimplement the algorithms. Their job is to choose settings, call the shared helpers, and explain the outputs.

In practice, the workflow is:

1. find the project root and import shared modules
2. build one or two shared config bundles
3. choose experiment grids such as feature counts or seeds
4. run a helper like `run_falp_grid(...)` or `run_psmd_seed_grid(...)`
5. visualize the stored results with plotting helpers
6. interpret the lower bounds, policy costs, and gaps


In [ ]:
print('Notebook files:')
for path in sorted((PROJECT_ROOT / 'notebooks').glob('*.ipynb')):
    print(' ', path.name)


The three notebooks have different roles:

- `self-guided-alp.ipynb`: main FALP and SGALP tutorial
- `psmd.ipynb`: PSMD baseline tutorial
- `how-code-works.ipynb`: codebase tour and orientation notebook


---
<a id="dependencies"></a>
## 11. Dependency Picture

```mermaid
flowchart TD
    A[config.py] --> B[mdp.py]
    A --> C[basis.py]
    A --> D[self_guided_alp/falp.py]
    A --> E[self_guided_alp/sgalp.py]
    A --> F[psmd/psmd.py]
    B --> D
    B --> E
    B --> F
    C --> D
    C --> E
    C --> F
    D --> G[self_guided_alp/cvl_lower_bound.py]
    E --> G
    D --> H[policy.py]
    E --> H
    F --> G
    F --> H
    G --> I[helper.py]
    H --> I
    D --> I
    E --> I
    F --> I
    I --> J[notebooks/*.ipynb]
```

The important idea is that `helper.py` sits close to the notebooks, not at the mathematical core. The mathematical core is the MDP, basis, fitted models, and evaluators. The helpers organize those pieces into tutorial-friendly experiments.


---
<a id="reading-order"></a>
## 12. Reading Order

A practical reading order for this project is:

1. `config.py`
   Learn what choices the project exposes.
2. `mdp.py`
   Learn the inventory problem and the shared sampling interface.
3. `basis.py`
   Learn how value functions are represented.
4. `self_guided_alp/falp.py` and `self_guided_alp/sgalp.py`
   Learn how the ALP approximations are fitted.
5. `policy.py` and `self_guided_alp/cvl_lower_bound.py`
   Learn how fitted models are evaluated.
6. `psmd/psmd.py`
   See the baseline method that reuses the same reporting ideas.
7. `helper.py`
   Learn how the notebooks package experiments and figures.
8. `notebooks/*.ipynb`
   See the final user-facing workflow.

If you keep one sentence in mind while reading, this is the one:

**the project is easier than it first looks because most files have exactly one job.**
